# 2. Machine Learning for Regression

**Car Price Prediction Project**

Session two is one hands-on project: we build a model that predicts the price of a car. A user wants to sell a car on an online classified website, and the site asks them to enter a price. If they don't know how much the car is worth, they have to guess. We want to help: the user describes the car, and our model suggests the best price.

**Project plan:**

1. [Data Preparation](#2.2-Data-Preparation)
2. [Exploratory Data Analysis (EDA)](#2.3-Exploratory-Data-Analysis)
3. [Validation Framework](#2.4-Validation-Framework)
4. [Linear Regression — Simple Form](#2.5-Linear-Regression)
5. [Linear Regression — Vector Form](#2.6-Linear-Regression:-Vector-Form)
6. [Training Linear Regression: Normal Equation](#2.7-Training-Linear-Regression:-Normal-Equation)
7. [Baseline Model](#2.8-Baseline-Model)
8. [Root Mean Squared Error (RMSE)](#2.9-Root-Mean-Squared-Error)
9. [Using RMSE on Validation Data](#2.10-Using-RMSE-on-Validation-Data)
10. [Feature Engineering](#2.11-Feature-Engineering)
11. [Categorical Variables](#2.12-Categorical-Variables)
12. [Regularization](#2.13-Regularization)
13. [Tuning the Model](#2.14-Tuning-the-Model)
14. [Using the Model](#2.15-Using-the-Model)
15. [Summary](#2.16-Summary)
16. [Explore More](#2.17-Explore-More)

---


## 2.1 Car Price Prediction Project

Session two is one hands-on project: we build a model that predicts the price of a car.

### The scenario

Remember the scenario from the introduction. A user wants to sell a car on an online classified website, and the site asks them to enter a price. If they don't know how much the car is worth, they have to guess. We want to help: the user describes the car, and our model suggests the best price.

### The dataset

To train a model we need data, and we use a dataset with car prices from Kaggle. It has different characteristics of cars — the make (the manufacturer), the model, the year, the engine type, the fuel type and other properties.

The data explorer on Kaggle shows what is inside. Every row is one car, and every column is one characteristic of that car.

One column is especially interesting for us: **MSRP**. It stands for Manufacturer Suggested Retail Price — in other words, the price of a car. This is exactly what we want to predict. The plan is to use all the other features — the make, the model, the year, the engine and so on — to predict this price.

### The project plan

We will do the project in several steps:

- Get the data and do **exploratory data analysis**: just look at the data and try to learn more about it.
- Prepare the dataset and train a **linear regression model** on it to predict the price of a car.
- Go into the details of how linear regression is implemented — we will actually implement it ourselves.
- Evaluate the quality of the model with **RMSE** (root mean squared error).
- Do some **feature engineering**: the process of creating new features, new characteristics that we can use for our model.
- Deal with **numerical stability** problems and see how to solve them with regularization.
- Finally, **use the model**.

The dataset is available at: https://github.com/alexeygrigorev/mlbookcamp-code/tree/master/chapter-02-car-price

---


## Imports and Setup

We start by importing the libraries we'll need throughout this session.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

---

## 2.2 Data Preparation

Now we take the car price dataset and prepare it for machine learning: we download it, load it with pandas, and make the column names and the string values consistent.


### Getting the data

The dataset lives on Kaggle, but there is a copy in the mlbookcamp-code repository. To get it, we can use `wget` or download it directly from the URL. If you already have `data.csv` locally, skip this cell:


In [ ]:
data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv'
!wget $data_url -O data.csv

### Loading the data

To load a CSV file in pandas we use the `read_csv` function. It reads the data and returns a DataFrame:


In [ ]:
df = pd.read_csv('data.csv')
df.head()

We see the manufacturer of the car, the model, the year and a lot of other characteristics. And **MSRP** — the manufacturer suggested retail price — is what we want to predict.

### Making the column names consistent

Looking at the column names, there is some inconsistency. Sometimes there are uppercase letters (`Make`, `Model`), sometimes there are spaces (`Engine Fuel Type`), sometimes underscores (`Driven_Wheels`).

Spaces are the annoying part. Usually, if we want to access the transmission type column, we cannot use dot notation — `df.Transmission Type` is a syntax error. We have to use bracket notation instead, `df['Transmission Type']`. To keep life simple, let's clean the names: make everything lowercase and replace spaces with underscores.

In pandas, every DataFrame has the `columns` attribute, which contains the names of the columns. It is an index — a special data structure in pandas, very similar to a series — and like a series it has the `str` accessor for doing string manipulations. So we can apply a string function to all the column names at once:


In [ ]:
# Preview the transformation
df.columns.str.lower().str.replace(' ', '_')

This alone changes nothing — we need to write the result back to the `columns` attribute:


In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Now the column names are uniform: all lowercase, no spaces. Cleaner.

### Finding the string columns

The values have the same problem: sometimes they are all caps (`BMW`, `MANUAL`), sometimes not. Let's normalize the string values the same way.

First we need to find out which columns are strings — we cannot apply string methods to numbers. For that we use the `dtypes` attribute:


In [ ]:
df.dtypes

The type we are interested in is `object` — in pandas, that is how strings are stored. To select only the columns of type `object`, we compare `dtypes` with `'object'`. What we want are the names, so we take the index of this series and convert it to a Python list:


In [ ]:
strings = list(df.dtypes[df.dtypes == 'object'].index)
strings

### Normalizing the string values

Now that we have all the string column names, we loop over them and apply the same transformation we used for the column names: lowercase everything and replace spaces with underscores, writing the result back to the DataFrame:


In [ ]:
for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

df.head()

After the loop the data looks cleaner. Everything is lowercase, and spaces in values are replaced with underscores.

**Key pandas methods used:**
- `pd.read_csv(file_path)` — read CSV files
- `df.head()` — take a look at the first rows of the dataframe
- `df.columns` — retrieve column names of a dataframe
- `df.columns.str.lower()` — lowercase all the letters
- `df.columns.str.replace(' ', '_')` — replace the space separator
- `df.dtypes` — retrieve data types of all features
- `df.index` — retrieve indices of a dataframe

---


## 2.3 Exploratory Data Analysis

In the previous lesson we prepared the columns of our DataFrame. Now we do exploratory data analysis (EDA): we look at every column, see what kind of values are there, and try to get a feeling for the data and the problem before training a model.

### Looking at every column

We already know that some columns are strings and some are numbers. To see what is inside each of them, we iterate over all columns and print some statistics for every one: the column name, a few values, and the number of unique values:


In [ ]:
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print()

`unique()` returns the unique values in a column — here we take the first five. `nunique()` returns how many unique values there are.

Walking through the output: `make` is the manufacturer of a car — BMW, Audi, Fiat, Mercedes-Benz, Chrysler, 48 of them in total. `model` is more granular: every manufacturer has multiple models, which is why there are 914 unique values. `year` is a numerical column with 28 different years. `engine_fuel_type` is the type of fuel the engine gets — diesel and others, ten types in total.

`highway_mpg` is how many miles the car can drive on a highway per gallon of fuel, and `city_mpg` is the same in the city. `popularity` is something the authors of the dataset computed: they looked at Twitter and counted how many mentions a car had. And finally `msrp`: the price of a car — this is what we want to predict.

### Distribution of price

The price column deserves a closer look. Looking at numbers is not very informative — we don't see the big picture. So we visualize it.

We want to see the distribution of prices — how many cars cost what. A histogram shows exactly that: it splits the value range into buckets and draws one bar per bucket, with the height showing how many values fall into it:


In [ ]:
sns.histplot(df.msrp, bins=50)
plt.title('Distribution of car prices (MSRP)')
plt.xlabel('Price')
plt.show()

The `1e6` on the x axis is scientific notation: 10 to the power of 6, that is, one million. And what we see is that a lot of prices are pretty cheap — most of the cars are concentrated on the left. Then there are very few cars that are super expensive.

This kind of distribution is called a **long-tail distribution**: most of the data sits in one place, and a shallow tail stretches far to the right. Long-tail distributions are very common for prices.

To see the shape better, we zoom in on prices below 100,000:


In [ ]:
sns.histplot(df.msrp[df.msrp < 100000], bins=50)
plt.title('Distribution of car prices below $100,000')
plt.xlabel('Price')
plt.show()

This is the left part of the previous histogram, and it is much easier to read. There is a strange peak of cars that cost 1,000 — probably the minimal price possible to put on the platform. Then the number of cars slowly grows up to around 25,000 — about 700 cars cost that much — and after that it goes down as the price grows.

### Removing the long tail with a logarithm

While this distribution is not unexpected, it is not really good for machine learning: the long tail will confuse the model and screw things up. We want to get rid of it, and the usual trick is to apply the **logarithm** to the price.

The logarithm compresses large values. For large numbers, the value of the logarithm is not that large: going from 10 to 1,000 is a big jump, but the increase between their logarithms is not that high.

There is one problem with the plain logarithm: the logarithm of zero doesn't exist. In our case prices are always 1,000 or more, so this can't happen. Still, it's pretty common to add one to all values before taking the logarithm, just to be safe. NumPy has a shortcut for exactly this: `log1p`, where "1p" stands for "plus one":


In [ ]:
# Demo: log1p adds 1 then takes the log
np.log1p([0, 1, 10, 1000, 100000])

In [ ]:
# Apply log1p to prices
price_logs = np.log1p(df.msrp)

sns.histplot(price_logs, bins=50)
plt.title('Distribution of log(price + 1)')
plt.xlabel('log(MSRP + 1)')
plt.show()

The tail is gone. All the large prices collapsed into a small area on the right, and the cars for usual consumers are still concentrated around the center. The shape now resembles a bell curve: a clear center of the distribution that goes down on both the left and the right side. This is called a **normal distribution**.

This situation is ideal for models: models do much better when the target variable looks like a normal distribution than with long-tail ones. That is why we typically want to get rid of long tails, and applying the logarithm to the price, like we just did, is one way of doing it.

### Missing values

One more thing before we finish: missing values. Some values in the dataset were not recorded. When pandas doesn't have a value, it shows it as `NaN` — not a number. To count how many missing values there are, we use `isnull()`:


In [ ]:
df.isnull().sum()

For quite a few cars we don't know the fuel type, the market category, the horsepower or the number of cylinders. We need to keep this in mind: before training the model, we will need to do something with these missing values.

**Key methods:**
- `df[col].unique()` — return a list of unique values in the series
- `df[col].nunique()` — return the number of unique values in the series
- `df.isnull().sum()` — return the number of null values in the dataframe
- `sns.histplot()` — show the histogram of a series
- `np.log1p()` — apply log transformation to a variable, after adding one to each input value

> **Note:** Long-tail distributions usually confuse ML models, so the recommendation is to transform the target variable distribution to a normal one whenever possible.

---


## 2.4 Setting Up the Validation Framework

The dataset is cleaned and explored, so now we set up the validation framework: we split the data into train, validation and test parts, and prepare the target variable for each part. We do it manually, with plain pandas and NumPy.

### Train, validation and test

From the previous lesson we know why we need to validate models. To do it, we take our dataset and split it into three parts. The first part we use for training, the second one for validation, and the last one for testing. We train a model, check if it works fine on the validation dataset, and leave the test dataset alone: we use it only very occasionally, at the very end, to check that our model is doing good.

Then, from each of these parts we create a feature matrix **X** and the target variable **y**.

We will do a **60/20/20** split.

### Calculating the sizes of the parts


In [ ]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

n, n_val, n_test, n_train

Ideally we would write `n_train = int(n * 0.6)` as well, but because of rounding, the three parts may stop adding up to `n` — and we would accidentally leave a few records out. So instead, we take validation and test out first, and whatever remains goes to training.

### Shuffling the records

We need to break any order in the dataset. In `iloc` we can pass an arbitrary sequence of numbers, and it will return the rows in that order. So we can take the numbers from 0 to n-1, reshuffle them, and then take the first 20% of the shuffled numbers for validation, the next 20% for test, and the rest for train.

To make the results reproducible, we set a random seed before shuffling:


In [ ]:
np.random.seed(2)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val   = df.iloc[idx[n_train:n_train+n_val]]
df_test  = df.iloc[idx[n_train+n_val:]]

len(df_train), len(df_val), len(df_test)

### Resetting the index

Looking at `df_train`, we see that the index now contains the original row numbers — 2735, 6720 and so on, in random order. Sometimes it is inconvenient, so we reset the index:


In [ ]:
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

### Preparing y and removing msrp

The dataframes will be used to make the feature matrix X. The target y comes from the `msrp` column, and we already know that we need to apply the `log1p` transformation to it because of the long tail in its distribution. Instead of keeping a pandas series, we immediately take its values with `.values` to get a NumPy array:


In [ ]:
y_train = np.log1p(df_train.msrp.values)
y_val   = np.log1p(df_val.msrp.values)
y_test  = np.log1p(df_test.msrp.values)

The last thing we need to do is remove `msrp` from the dataframes, for which we use the `del` operator:


In [ ]:
del df_train['msrp']
del df_val['msrp']
del df_test['msrp']

We delete it because we might accidentally use it. If the price gets into the features, we will use the price to predict the price — and of course the model will be perfect. This happened to me many times, so after splitting the data I always take the target out into a separate variable and delete it from the dataframe completely.

**Key methods:**
- `df.iloc[]` — return subsets of records of a dataframe, being selected by numerical indices
- `df.reset_index()` — restate the original indices
- `del df[col]` — eliminate a column variable
- `np.arange()` — return an array of numbers
- `np.random.shuffle()` — return a shuffled array
- `np.random.seed()` — set a seed for reproducibility

---


## 2.5 Linear Regression

In this lesson we look at linear regression, the model we use for solving regression problems — problems where we predict a number, like the price of a car. We derive the formula, implement it in Python for one car, and see what the prediction is made of.

### The formula for one car

Linear regression is a model for solving regression problems. The model g will be linear regression:

$$g(X) \approx y$$

Here **g** is our model, **X** is the feature matrix and **y** is our target — the price.

Before we go to the matrix form, let's simplify and look at one observation instead of the entire feature matrix. One observation is one car:

$$g(x_i) \approx y_i$$

Let's take an example from the training dataset. Let's take row number 10. It is a Rolls-Royce Phantom Drophead Coupe manufactured in 2015, and it has a lot of characteristics. We take just three of them: engine horsepower, city miles per gallon and popularity:


In [ ]:
# Let's look at row 10 from the training data
df_train.iloc[10][['engine_hp', 'city_mpg', 'popularity']]

For our example, let's use the feature vector from the Rolls-Royce:


In [ ]:
xi = [453, 11, 86]  # engine_hp=453, city_mpg=11, popularity=86

### The linear regression formula

We need to combine these values in a way that gives us something close to the price. The formula for linear regression is:

$$g(x_i) = w_0 + w_1 \cdot x_{i1} + w_2 \cdot x_{i2} + w_3 \cdot x_{i3}$$

First we have the **bias term** $w_0$ — this is the prediction we make without knowing anything about the car. But we do know something, so each feature is multiplied by its weight.

The part with the features is a sum, so we can write the formula more compactly. Because we already use i for the car, we use j for the features, and it goes from 1 to n (in our example, 3):

$$g(x_i) = w_0 + \sum_{j=1}^{n} w_j \cdot x_{ij}$$

### Implementing it in Python

Let's implement this. We have the feature vector xi, the bias term w0, and a vector w with a weight for each feature. (We'll talk about where these weights come from later — for now, we came up with them ourselves.)


In [ ]:
xi = [453, 11, 86]
w0 = 7.17
w  = [0.01, 0.04, 0.002]

def linear_regression(xi):
    n = len(xi)
    pred = w0
    for j in range(n):
        pred = pred + w[j] * xi[j]
    return pred

The formula says the sum goes from 1 to n, but in Python indexes start at 0, so our loop goes from 0 to n-1. That's the whole implementation.

### What the prediction is made of


In [ ]:
linear_regression(xi)

To make sense of this number, let's look at the parts it is made of:

```
7.17 + 453·0.01 + 11·0.04 + 86·0.002 = 12.312
```

- We start with the bias term, **7.17**. This is what we predict for an average car if we don't know anything about it.
- Then we know the car has 453 horsepower. Each horsepower adds 0.01 to the prediction: the more horsepower the engine has, the more expensive the car becomes — that's logical.
- The next feature is miles per gallon in the city, with weight 0.04. The more fuel the car consumes in the city, the more expensive it is. This can make sense: cars that consume more fuel are probably fancier.
- The last one is popularity, the number of mentions on Twitter, with a pretty low weight of 0.002. It doesn't seem to affect the price much.

### From log price back to dollars

The prediction `12.312` is not the price in dollars. Remember that we applied the `log1p` transformation to y because of the long tail in the price distribution, so our model predicts the **logarithm** of the price. To undo the logarithm we need the exponent:


In [ ]:
# np.expm1 is the inverse of np.log1p
np.expm1(12.312)

So our prediction for this car is about 222,000 dollars. And there is a shortcut: `np.expm1` is the inverse of `np.log1p` — apply one after the other and you are back where you started:


In [ ]:
# Verify: log1p is the inverse of expm1
np.log1p(222347.2221101062)

That's linear regression for one car: we implemented the formula on a small feature vector of size three.

---


## 2.6 Linear Regression: Vector Form

Last time we implemented linear regression for a single car. Now we generalize it: first to a dot product for one car, then to a matrix-vector multiplication that produces predictions for all the cars in the dataset at once.

### The dot product

Let's again write the formula for a single car first. We have the bias term and then a sum that goes from 1 to n, where we multiply each feature with its weight:

$$g(x_i) = w_0 + \sum_{j=1}^{n} x_{ij} \cdot w_j$$

If you look at the sum, you can see that it is nothing else but a **dot product** — a vector-vector multiplication. Using that notation we can rewrite the formula slightly differently: we take the vector $x_i$, transpose it, and multiply it with the vector of weights:

$$g(x_i) = w_0 + x_i^T w$$

This notation is more compact, so let's implement it. We already have the `linear_regression` function; let's first extract the sum into a function of its own:


In [ ]:
def dot(xi, w):
    n = len(xi)
    res = 0.0
    for j in range(n):
        res = res + xi[j] * w[j]
    return res

def linear_regression(xi):
    return w0 + dot(xi, w)

### The fictional feature

We can make it even shorter. The bias term $w_0$ is kind of hanging out there all alone, without a feature. Let's bring it into the dot product: we imagine that there is one extra feature of our car, $x_{i0}$, and this feature is **always 1**.

Then the vector of weights becomes $w_0, w_1, w_2, \ldots, w_n$ — it is an n+1 dimensional vector now. And the vector of features becomes $x_{i0}, x_{i1}, x_{i2}, \ldots$, where $x_{i0} = 1$.

Why does this work? When we do the dot product, $w_0$ gets multiplied by 1, so it simply stays there, and the rest is the same dot product as before.

In code, in Python, adding two lists with `+` creates a new list and prepends the first one at the beginning — exactly what we need:


In [ ]:
xi   = [453, 11, 86]
w0   = 7.17
w    = [0.01, 0.04, 0.002]

w_new = [w0] + w
print('w_new:', w_new)

In [ ]:
def linear_regression(xi):
    xi = [1] + xi
    return dot(xi, w_new)

# Should give the same result as before: 12.312
linear_regression(xi)

### Linear regression for all cars

Now let's go back to thinking about all the examples, not just one. We have the matrix **X**. Because of the fictional feature, each row of this matrix starts with 1, followed by the features of that car.

For us X has m rows and n+1 columns. What we need to do is take each row of this matrix, do the dot product of that row with the vector of weights w, and that gives us the prediction for that car. Doing this for all the rows gives us the vector of predictions **y**.

This looks very similar to matrix-vector multiplication — in fact, it is exactly that:

$$Xw \approx y$$

Let's implement it with a small example of three cars:


In [ ]:
x1  = [1, 148, 24, 1385]
x2  = [1, 132, 25, 2031]
x10 = [1, 453, 11, 86]

X = np.array([x1, x2, x10])
print(X)

In [ ]:
w0    = 7.17
w     = [0.01, 0.04, 0.002]
w_new = [w0] + w

def linear_regression(X):
    return X.dot(w_new)

linear_regression(X)

For each of the three cars we get a prediction — the price for that car.

To summarize how we got here: we started with a for loop for one car, recognized that it is a dot product between the vector of features and the vector of weights, made it even shorter by adding the fictional feature — the virtual 1 — to the feature vector, and finally generalized it to a complete feature matrix, where linear regression is nothing else but a **matrix-vector multiplication**.

> **Summary:** The formula of linear regression can be synthesized with the dot product between features and weights. The feature vector includes the *bias* term with an *x* value of one, such that $w_0 = w_0 \cdot x_{i0}$ where $x_{i0} = 1$. When all the records are included, the linear regression can be calculated with the dot product between the **feature matrix** and **vector of weights**, obtaining the `y` vector of predictions.

---


## 2.7 Training Linear Regression: Normal Equation

In the previous unit we learned how to apply a linear regression model when we already have the weights. This unit answers the remaining question: **where do the weights come from?** We derive the normal equation — a closed-form formula for the weights — and implement it as a `train_linear_regression` function.

### Solving for the weights

We have the feature matrix `X`, and we want the predictions $g(X) = Xw$ to be close to the actual target values `y`:

$$Xw \approx y$$

Ideally, $Xw$ equals $y$ exactly, and we can try to solve this system for `w`.

Suppose for a moment that `X` is invertible: there exists a matrix $X^{-1}$ such that $X^{-1}X$ is the identity matrix $I$. Then we can multiply both sides of the equation by $X^{-1}$:

$$X^{-1}Xw = X^{-1}y \implies Iw = X^{-1}y \implies w = X^{-1}y$$

The inverse "cancels" `X`, and we get the weights directly.

There is a problem, however. `X` is a rectangular matrix: it has $m$ rows (one per car) and $n+1$ columns (one per feature, plus the bias term). In our dataset $m$ is much larger than $n+1$, so the matrix is definitely not square — and only square matrices can have an inverse. The solution to this system simply doesn't exist.

### The Gram matrix and the normal equation

Even though the exact solution doesn't exist, we can find the closest possible approximate solution. Multiply both sides of $Xw \approx y$ by the transpose of `X`:

$$X^T X w = X^T y$$

The matrix $X^T X$ is called the **Gram matrix**. Unlike `X`, the Gram matrix is square — it has $n+1$ rows and $n+1$ columns. Square matrices usually have an inverse, so we can multiply both sides by $(X^T X)^{-1}$:

$$(X^T X)^{-1} X^T X w = (X^T X)^{-1} X^T y$$

On the left, $(X^T X)^{-1} X^T X$ is the identity matrix, and $Iw = w$. That leaves us with the **Normal Equation**:

$$\boxed{w = (X^T X)^{-1} X^T y}$$

This $w$ is not the solution to the original system — that solution doesn't exist — but it is the closest possible solution.

### Implementing the normal equation

Let's implement the formula step by step in NumPy. To keep things simple, we use a small toy matrix first: nine rows and three features:


In [ ]:
X = np.array([
    [148, 24, 1385],
    [132, 25, 2031],
    [453, 11, 86],
    [158, 24, 185],
    [172, 25, 201],
    [413, 11, 86],
    [38,  54, 185],
    [142, 25, 431],
    [453, 31, 86],
])

# Step 1: The Gram matrix
XTX = X.T.dot(X)
XTX

In [ ]:
# Step 2: Its inverse
XTX_inv = np.linalg.inv(XTX)

# Sanity check: XTX * XTX_inv should give the identity matrix
XTX.dot(XTX_inv).round(1)

We see ones on the diagonal and values that are extremely close to zero elsewhere. They are not exactly zero because floating-point numbers have finite precision, so small round-off errors accumulate — but we can safely treat them as zero.

Now we need a target vector `y`. Let's just come up with nine prices on the spot:


In [ ]:
y_toy = [100, 200, 150, 250, 100, 200, 150, 250, 120]

# Step 3: Apply the normal equation (without bias term yet)
w_full = XTX_inv.dot(X.T).dot(y_toy)
w_full

With this matrix we actually forgot one thing: the **bias term**. The bias is important — it's the baseline prediction, the price of a car when we know nothing else about it. To include it, we add a column of ones to `X` and repeat the same steps:


In [ ]:
ones = np.ones(X.shape[0])
X_with_ones = np.column_stack([ones, X])

XTX = X_with_ones.T.dot(X_with_ones)
XTX_inv = np.linalg.inv(XTX)
w_full = XTX_inv.dot(X_with_ones.T).dot(y_toy)

# Split bias term from feature weights
w0_toy = w_full[0]
w_toy  = w_full[1:]

print('Bias (w0):', w0_toy)
print('Weights  :', w_toy)

The `np.column_stack` function stacks the vector of ones and the feature matrix together as columns, so the ones become the first column of `X`.

Note that the weights can be negative. A negative weight means: as the feature value grows, the predicted price goes down. If this feature were the age of a car, that would make perfect sense: the older the car, the lower the price.

### Wrapping into a function

Finally, let's wrap everything into a function. The function takes `X` and `y`, adds the column of ones internally, applies the normal equation, and returns the bias term and the weights separately:


In [ ]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

Because adding the ones happens inside the function, users of it don't need to worry about the bias term: they pass the plain feature matrix, and the function takes care of the rest.

> **Summary:** The Normal Equation gives us a closed-form solution for the weights of linear regression:
> $$w = (X^T X)^{-1} X^T y$$
> where $X^T X$ is the **Gram Matrix**. This is the closest possible solution to the system $Xw \approx y$.

---


## 2.8 Baseline Model for Car Price Prediction

In the previous unit we implemented `train_linear_regression`, the function that finds the weights of a linear regression model. Now we finally use it on the car price data to build our first model: a **baseline**. It is deliberately simple — only the numerical features — and its quality is the reference point we will try to improve on.

### Selecting the base features

First, let's remind ourselves which columns the training dataframe has:


In [ ]:
df_train.columns

Linear regression as we implemented it works only with numbers, so for the baseline we pick the numerical columns:

- `engine_hp` — engine horsepower
- `engine_cylinders` — number of cylinders
- `highway_mpg` — miles per gallon on the highway
- `city_mpg` — miles per gallon in the city
- `popularity`


In [ ]:
base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity']

# Get the feature matrix
X_train = df_train[base].values

### Dealing with missing values

Let's try to train the model right away:


In [ ]:
# First check for missing values
df_train[base].isnull().sum()

We see that `engine_hp` and `engine_cylinders` contain missing values. The easiest way to deal with them is to **fill them with zeros**:


In [ ]:
X_train = df_train[base].fillna(0).values

Filling with zeros might look strange — a car with zero horsepower doesn't exist, and no engine has zero cylinders. To see why it still works, look at what a missing value does in the linear regression formula. Say we have two features, and the first one is missing:

```
g(xi) = w0 + xi1·w1 + xi2·w2      (xi1 is missing)
     = w0 +   0·w1 + xi2·w2
     = w0 +          xi2·w2
```

When `xi1` is zero, its term `xi1 · w1` disappears, so the model effectively ignores that feature. From a common-sense point of view, replacing a value with the mean of that feature is often more reasonable — but from a practical point of view, zeros keep the process simple, and sometimes they work just fine.

### Training the model

Now the feature matrix has no missing values, and training finishes:


In [ ]:
X_train = df_train[base].fillna(0).values
w0, w = train_linear_regression(X_train, y_train)

print('Bias (w0):', w0)
print('Weights  :', w)

The bias term comes out as `w0 ≈ 7.93`. Remember that we predict the **logarithm** of the price, so this is on the log scale. Together with the five weights, this is our baseline model.

### Comparing the predictions with the actual prices

Now we can apply the trained model to the training data. We use matrix multiplication, adding the bias term separately:


In [ ]:
y_pred = w0 + X_train.dot(w)

In [ ]:
# Plot predictions vs. actual values on the same histogram
sns.histplot(y_pred,   color='red',  alpha=0.5, bins=50, label='prediction')
sns.histplot(y_train,  color='blue', alpha=0.5, bins=50, label='actual')
plt.legend()
plt.title('Baseline model: predictions vs. actual prices (log scale)')
plt.xlabel('log(price + 1)')
plt.show()

Looking at the chart, the distribution of predictions does not really match the distribution of actual values: the red peak sits to the left of the blue one, so in many cases the model predicts a value smaller than the actual price.

Just by looking at this chart we suspect the model is not ideal, but that is not an objective way to say it. And when we start improving the model, we want to be sure that it indeed improved. For that we need a single number that quantifies how good or bad the model is. That is exactly what we will do in the next unit with RMSE.

---


## 2.9 Root Mean Squared Error (RMSE)

In the previous unit we trained our first model and plotted the predictions against the actual values. The histograms showed the predictions are a bit off, but a chart is not an objective measure. In this unit we learn one way of quantifying how good or bad a regression model is: **root mean squared error**, or RMSE.

### The formula

RMSE starts from the difference between the prediction and the actual value. For each observation $i$, $g(x_i)$ is the prediction we make and $y_i$ is the actual value. We take the difference, square it, average these squared differences over all $m$ observations, and finally take the square root:

$$RMSE = \sqrt{\frac{1}{m} \sum_{i=1}^{m} (g(x_i) - y_i)^2}$$

### A worked example

Imagine we have four observations. These are our predictions, and these are the actual prices:

```
y_pred:  10     9     11    10
y:        9     9     10.5  11.5
```

**Step 1: Differences**
```
10 - 9    =  1
 9 - 9    =  0
11 - 10.5 =  0.5
10 - 11.5 = -1.5
```

**Step 2: Square each difference**
```
1    → 1
0    → 0
0.5  → 0.25
-1.5 → 2.25
```

Squaring does two things: it makes negative and positive errors comparable — otherwise they would cancel each other out — and it penalizes larger errors more.

**Step 3: Average** (Mean Squared Error, MSE)
```
(1 + 0 + 0.25 + 2.25) / 4 = 0.875
```

**Step 4: Square root**
```
sqrt(0.875) ≈ 0.93
```

The square root brings the error back to the same units as the target variable.

### Implementation


In [ ]:
def rmse(y, y_pred):
    se  = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

First we compute the squared error — the squared difference between `y` and `y_pred`. To get the mean we don't need to sum and divide by the number of elements: NumPy has the `mean` method, which gives us the mean squared error directly. Finally, we take the square root and return the result.

Now let's use it to evaluate our baseline model. We already have the predictions `y_pred` on the training set:


In [ ]:
rmse(y_train, y_pred)

The RMSE of the baseline model is about **0.76**. This single number is much easier to work with than a chart: whenever we change something — add a feature, transform a variable, try a different model — we recompute it. If the number goes down, the model got better.

So far we computed it on the training data. In the next unit we apply the model to the validation set and see how it performs there.

> **Key formula:**
> $$RMSE = \sqrt{\frac{1}{m} \sum_{i=1}^{m} (g(x_i) - y_i)^2}$$
> where $g(x_i)$ is the prediction, $y_i$ is the actual value, and $m$ is the number of observations.

---


## 2.10 Using RMSE on Validation Data

In the previous unit we implemented RMSE and used it to measure the quality of our baseline model. There was one problem: we measured it on the **training data** — the same data the model learned from. Here we fix that and evaluate the model the way it should be evaluated: on the **validation data**.

### The validation framework

Remember how validation works. We take the dataset and split it into three parts: the training set, the validation set and the test set. We train the model on the training part, and then we apply it to the validation part to see how it performs on data it has never seen.

### Preparing the feature matrix with `prepare_X`

Let's take the code we wrote earlier. A lot of things are happening in one line, so let's write a special function for that. We'll call it `prepare_X`, and it takes a dataframe:


In [ ]:
def prepare_X(df):
    df_num = df[base]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

The function does three things. First it selects the numerical columns, then it fills the missing values, and finally it extracts the feature matrix — the NumPy array — and returns it.

The point of this function is that we prepare the data in the same way regardless of whether it is the training, validation or test dataset. It works on any dataframe — that is why the argument is just `df` and not `df_train`.

### Computing the RMSE on validation data

Now we can train the model and validate it. First the training part: we prepare the matrix from the training dataframe and train the model. Then the validation part: we prepare the validation matrix with the same function, apply the model, and compute the RMSE:


In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
print('Validation RMSE:', rmse(y_val, y_pred))

The number is pretty similar to what we had on the training data. That is what we want to see: the model behaves on unseen data about as well as on the data it learned from.

Looking at the code, we can see the two parts clearly:

- The **training part** only touches the training dataset: we prepare the matrix and learn the weights.
- The **validation part** prepares the validation dataset in the same way as the training dataset, then applies the model learned in the previous step, and computes the root mean squared error for that model.

Now we have a way of evaluating the quality of our model using the root mean squared error on the validation dataset. This also means we can check whether a change actually improves the model — and this is what we will do next, starting with feature engineering.

---


## 2.11 Feature Engineering

Feature engineering is the process of creating new features from the ones we already have. In this unit we create our first new feature — the **age of a car** — and see it improve the RMSE of our model quite a bit.

### From year to age

Let's take a look again at our training dataset. One of the columns there is `year`, and we know this is one of the most important variables for predicting the price of a car: if a car is old, it is usually cheaper, and if it is new, it is more expensive.

Instead of using the year as it is, we can compute the **age** of a car. For that we need to know when this data was collected — it turns out it was collected in 2017. So "now" is 2017, and the age is simply `2017 - year`:


In [ ]:
# Preview the age feature
(2017 - df_train.year).head(10)

This age is what we want to use as a feature in our model.

### Adding the age feature to `prepare_X`

Let's modify the `prepare_X` function we wrote in the previous unit. The new feature is called `age`, and we compute it in the same way we just did.

We also need to use the new feature when building the matrix. Instead of `base`, we create a list called `features` that contains the baseline numerical features plus the new `age` feature.

**Important:** The function should **not modify the data**. When we run this function, the first line adds a new column — and if we look at `df_train` again, we see that a new column appeared in it. We modified the dataframe we passed in.

The fix is simple: before doing anything, we take a **copy** of the dataframe and work with that copy inside the function. The original dataframe stays unchanged:


In [ ]:
def prepare_X(df):
    df = df.copy()

    df['age'] = 2017 - df.year
    features = base + ['age']

    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

Now `X_train` has six columns — the five base features plus `age`, which is the last one.

### Checking the improvement


In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
print('Validation RMSE (with age feature):', rmse(y_val, y_pred))

The model improved quite a bit — the RMSE went down from 0.76 to about **0.51**. That's a big improvement.

We can see that it is a big improvement by doing the same thing as previously: plotting the predicted values and the actual values on the same histogram:


In [ ]:
sns.histplot(y_pred, label='prediction', color='red',  alpha=0.5, bins=50)
sns.histplot(y_val,  label='target',     color='blue', alpha=0.5, bins=50)
plt.legend()
plt.title('With age feature: predictions vs. actual prices (log scale)')
plt.xlabel('log(price + 1)')
plt.show()

The shapes of the two distributions are now much closer. There is still a lot of room for improvement, but in general it is doing much better.

We can add even more features. In the next unit we will talk about **categorical variables** — columns like `model`, `make` and so on.

---


## 2.12 Categorical Variables

Categorical variables are variables that contain **categories** instead of numbers. In this unit we add them to our car price model and see that one of them breaks it completely — which is exactly the problem the next unit solves.

### Variables that are categories

So far we only used numerical features. But the dataset has quite a few categorical variables — variables that are categories. They are typically strings: `make`, `model`, `engine_fuel_type`, `transmission_type`, `driven_wheels` and others. If we look at the data types, all the columns of type `object` are categorical variables.

There is one variable that looks numerical but is not: `number_of_doors`. It contains 2, 3 and 4 — numbers — but these numbers are distinct categories of cars: cars with two doors, cars with three doors and cars with four doors are quite different.

### Encoding categories as binary columns (One-Hot Encoding)

ML models can only work with numbers, so we need to encode categorical variables. The typical way of doing it: for each value of the column, we create a separate binary column. If our column contains the values 2, 3, 4 and 2 again, we represent it with three columns — one for each door count — and put 1 in the column that matches the row, and 0 everywhere else:

| number_of_doors | num_doors_2 | num_doors_3 | num_doors_4 |
| --- | ---: | ---: | ---: |
| 2 | 1 | 0 | 0 |
| 3 | 0 | 1 | 0 |
| 4 | 0 | 0 | 1 |
| 2 | 1 | 0 | 0 |

One categorical column becomes multiple binary columns, and each row gets exactly one 1. This is called **one-hot encoding**.

In pandas, the comparison operator does most of the work. When we say `df.number_of_doors == 2`, we get `True` for every car that has two doors. The result is a series of booleans, so the only thing left is to turn it into integers — ones and zeros — with `astype(int)`:


In [ ]:
# Preview: encoding number_of_doors
(df_train.number_of_doors == 2).astype('int').head()

### Adding doors to `prepare_X`

Let's modify our `prepare_X` function again. We make a copy of the dataframe as before, because we don't want to accidentally write these columns into our data. We also make a copy of the `base` list: if we used `append` on `base` itself, we would append `age` to it every time the function runs:


In [ ]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()

    df['age'] = 2017 - df.year
    features.append('age')

    for v in [2, 3, 4]:
        df['num_doors_%s' % v] = (df.number_of_doors == v).astype('int')
        features.append('num_doors_%s' % v)

    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
print('RMSE with doors feature:', rmse(y_val, y_pred))

It improved only slightly — the number of doors feature is not that useful.

### Adding the make

But I'm pretty sure that `make` should be quite useful. If we look at the values of this column, there are a lot of them — 48 unique makes. We can take the most popular ones with `value_counts`:


In [ ]:
df_train.make.value_counts().head()

In [ ]:
# Get the 5 most popular makes
makes = list(df_train.make.value_counts().head().index)
makes

This gives us the five most popular makes of cars: chevrolet, ford, volkswagen, toyota and dodge. We can include them in the same way as the number of doors — one binary column per make.

### Doing the same for all categorical variables

We can do the same for the other categorical variables: `engine_fuel_type`, `transmission_type`, `driven_wheels`, `market_category`, `vehicle_size` and `vehicle_style`. I'm not adding `model` here because there are simply too many models.

First we collect the top five most common values for each of them into a dictionary:


In [ ]:
categorical_variables = [
    'make', 'engine_fuel_type', 'transmission_type', 'driven_wheels',
    'market_category', 'vehicle_size', 'vehicle_style'
]

categories = {}

for c in categorical_variables:
    categories[c] = list(df_train[c].value_counts().head().index)

categories

Then we take `prepare_X` and throw in all the categories. We need two loops: one over the key-value pairs of the dictionary, and inside it another loop over the values of each category:


In [ ]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()

    df['age'] = 2017 - df.year
    features.append('age')

    for v in [2, 3, 4]:
        df['num_doors_%s' % v] = (df.number_of_doors == v).astype('int')
        features.append('num_doors_%s' % v)

    for c, values in categories.items():
        for v in values:
            df['%s_%s' % (c, v)] = (df[c] == v).astype('int')
            features.append('%s_%s' % (c, v))

    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
print('RMSE with all categorical features:', rmse(y_val, y_pred))

### Something went wrong

We execute the same validation code again, and now the value is significantly higher than what we had before. Before we had about 0.5, and now all of a sudden it's huge (around 41).

Something went wrong. If we look at the weights that our `train_linear_regression` function outputs, they are huge — some of them are around 10 to the power of 15. We wanted to improve our model by adding more variables, but we just made it worse.

In the next unit we will see why that happened and how to fix it.

---


## 2.13 Regularization

In the previous unit we added categorical variables to our model and ended up with huge weights. In this unit we look at why that happened and learn a technique for dealing with it: **regularization**.

### The inverse that doesn't always exist

Remember the formula for the normal equation we use for training our model:

$$w = (X^T X)^{-1} X^T y$$

The problem we have is with this part: we need to take the inverse of the Gram matrix $X^T X$, and sometimes this inverse doesn't exist.

It usually happens when the feature matrix X has **duplicate columns**. Let's quickly check it with a small matrix where the second and the third columns have the same values:


In [ ]:
X_dup = np.array([
    [4, 4, 4],
    [3, 5, 5],
    [5, 1, 1],
    [5, 4, 4],
    [7, 5, 5],
    [4, 5, 5],
], dtype=float)

# Compute the Gram matrix
XTX_dup = X_dup.T.dot(X_dup)
print('Gram matrix:')
print(XTX_dup)

When we compute the Gram matrix, we see that it also has duplicate columns — the second and the third are the same. When something like that happens, the inverse simply doesn't exist. In linear algebra they say that one column is a **linear combination** of the other columns — which here just means that the third column is a duplicate of the second one.

### Noisy data makes the inverse huge

This is not exactly the case for our car price problem — there we didn't see the singular matrix error. The reason is that real data is usually not super clean. When people record observations, they make mistakes: instead of writing 5, they write 5.00000001.

Let's add a tiny number like that to one value in our matrix:


In [ ]:
X_noisy = np.array([
    [4, 4, 4],
    [3, 5, 5],
    [5, 1, 1],
    [5, 4, 4],
    [7, 5, 5],
    [4, 5, 5.00000001],  # tiny noise
], dtype=float)

XTX_noisy = X_noisy.T.dot(X_noisy)
XTX_noisy_inv = np.linalg.inv(XTX_noisy)
print('Inverse (notice the huge numbers!):')
print(XTX_noisy_inv)

Now the columns are no longer exactly the same — they are slightly different — so the matrix is not exactly singular anymore. But look at what it comes up with: the numbers in the inverse are huge, around 10 to the power of 14.

When we use this inverse to compute the weights, the weights for the duplicated features turn out to be very large numbers too.

### Controlling the weights with regularization

To solve this problem, we can **add a small number to the diagonal of the matrix**. If we add a small number to the diagonal — say 0.01 — the inverse can still be found, and the numbers in it are much more under control. The larger the number we add to the diagonal, the more control we get over the weights.

This works because by adding something to the diagonal, we make sure that it is less possible that one column is a duplicate of another.

To add a number to the diagonal, we use the **identity matrix** — `np.eye(3)` is the 3×3 identity matrix, which has ones on the diagonal. When we multiply it by a small number and add it to our matrix, the diagonal increases by that small amount:


In [ ]:
# Small example: singular gram matrix
XTX_small = np.array([
    [1, 2, 2],
    [2, 1, 1],
    [2, 1, 1],
], dtype=float)

# Add regularization (0.01 * identity matrix)
r = 0.01
XTX_reg = XTX_small + r * np.eye(3)

print('Regularized XTX:')
print(XTX_reg)
print()
print('Inverse of regularized XTX:')
print(np.linalg.inv(XTX_reg).round(2))

This way of controlling the weights is called **regularization** — the name comes from controlling, regulating the weights so that they don't grow too much.

The number we add is actually a parameter. The larger the number on the diagonal, the smaller the values in the inverse of $X^T X$. How much regularization to add becomes a parameter of our model.

> **Linear combination** — I mentioned the term *linear combination* in the video. If you're interested what it means:
> - One column is a linear combination of others when you can express one column of a matrix as a sum of other columns
> - The simplest example is when a column is an exact duplicate of another column
> - Another example: if `c = 0.2 * a + 0.5 * b`, then `c` is a linear combination of `a` and `b`
>
> **Ridge Regression** — the regularization technique used here (adding a factor to the diagonal of the Gram Matrix) is known as Ridge Regression.

### The regularized training function

With that, we can go back to our function for training linear regression and slightly change it. We'll call it `train_linear_regression_reg`, and it gets one more parameter — `r`, short for regularization:


In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])   # <-- regularization

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

Everything stays the same except one line: after computing the Gram matrix, we add `r` to the main diagonal.

Let's take the code from the previous unit and replace the training function with this new one, using `r=0.01`:


In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression_reg(X_train, y_train, r=0.01)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
print('RMSE with regularization (r=0.01):', rmse(y_val, y_pred))

The result is not only much better than the huge RMSE we got with the non-regularized version — it is also better than what we had before adding all the categorical variables!

By adding a number to the diagonal we were able to control our weights and regularize the model. But `r` is a parameter: if we set it too high, the model becomes worse, and if we set it to 0, we are back to the usual linear regression. So now we need to find the best value for `r` — and this is what we will do in the next unit.

---


## 2.14 Tuning the Model

In the previous unit we saw that the regularization parameter `r` affects the quality of our model. **Tuning the model** means finding the best value for this parameter — and the validation set is exactly the tool for that.

### Trying different values of `r`

What we will do is simply try a bunch of different values for `r`: starting from zero, then something small, gradually increasing it, and maybe even 10:


In [ ]:
print(f'{'r':>10}  {'w0':>8}  {'RMSE':>10}')
print('-' * 35)

for r in [0.0, 0.00001, 0.0001, 0.001, 0.1, 1, 10]:
    X_train = prepare_X(df_train)
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)

    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)

    print(f'{r:>10.5f}  {w0:>8.4f}  {score:>10.4f}')

### Reading the results

What we see: for zero regularization the bias term is huge and the RMSE is huge too. That is the broken model with the duplicated columns problem. But for even a little bit of regularization it improves immediately, and after that the score doesn't really change that much. It starts to become a bit worse as we increase the regularization, and when it's 10 it's even worse.

Looking at the bias term, the more regularization we add, the smaller it is. It looks like maybe **0.001** is a good value, because the model hasn't started to degrade in performance there, and it's not too large. To be honest, it doesn't really matter here — it could be this one or the one next to it. We can just go with 0.001.

### Training the final model

Let's train our model once again with the selected value:


In [ ]:
r = 0.001
X_train = prepare_X(df_train)
w0, w = train_linear_regression_reg(X_train, y_train, r=r)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
score = rmse(y_val, y_pred)
print('Best model RMSE on validation:', score)

We selected the best regularization parameter, trained the model with it, and we saw that it works on the validation set.

Now what we need to do is check it on the test dataset as well — and this is what we will do in the next unit.

---


## 2.15 Using the Model

In the previous unit we found the best regularization parameter. In this unit we train the final model, check it on the test data, and then use it the way it would be used in practice: to predict the price of a single car.

### Training the final model

Remember that we split our dataset into three parts: train, validation and test. So far we trained the model on the training part and computed the RMSE on the validation part. For the final model we do something different: we **combine** the training and validation datasets into one — let's call it `full_train` — train the model on all of it, and then make the final evaluation on the test dataset.

The idea is simple: now that we have tuned all the parameters, we want to give the model as much data as possible for training, and keep the test set untouched for the final check. The RMSE on test shouldn't be too different from what we saw on validation.

We have two dataframes, `df_train` and `df_val`, and we want to combine them into one. In pandas there is a function for that — `concat`, short for concatenate:


In [ ]:
df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)

print('Shape of df_full_train:', df_full_train.shape)

In [ ]:
X_full_train = prepare_X(df_full_train)

# For y, NumPy has concatenate — and since arrays have no index,
# we don't need to reset anything
y_full_train = np.concatenate([y_train, y_val])

print('X_full_train shape:', X_full_train.shape)
print('y_full_train shape:', y_full_train.shape)

In [ ]:
# Train the final model on the full training data
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

print('Bias (w0):', w0)
print('Number of weights:', len(w))

This is our final model, and these are its weights.

### Checking the model on test data

To check the final model, we prepare the test dataset exactly in the same way as we did the validation dataset — we just replace validation with test:


In [ ]:
X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)
score = rmse(y_test, y_pred)
print('Test RMSE:', score)

The RMSE is almost the same as on validation — a very good sign. It means our model **generalizes well**: it didn't get this score just by chance.

### Predicting the price of a car

Now we can use the model the way we actually want to use it: to predict the price of a car. Imagine there is a car, we extract all its features, put the feature vector into the model — and it tells us the price.

For that we can take any car from the test dataset and pretend it's a new car. This is fine, because we didn't train our model on this data. Let's take a Toyota Sienna:


In [ ]:
car = df_test.iloc[20].to_dict()
car

This gives us a dictionary with all the information about the car.

Usually we don't get a dataframe when we want to make a prediction. In a real life scenario it could be a website or an app where people enter the values about the car, and the website sends a request with all this information to the model — the model replies back with the price. That's why we turn the car into a dictionary: this is how such requests usually look.

Our `prepare_X` function expects a dataframe, though. So we create a small dataframe with a single row — pandas can create a dataframe from a list of dictionaries, and our list contains just the one car from the request:


In [ ]:
df_small = pd.DataFrame([car])
X_small = prepare_X(df_small)

# Apply the model to this feature matrix with one row
y_pred_log = w0 + X_small.dot(w)
y_pred_log = y_pred_log[0]
print('Prediction (log scale):', y_pred_log)

In [ ]:
# Undo the logarithm to get the price in dollars
predicted_price = np.expm1(y_pred_log)
print('Predicted price: $%.2f' % predicted_price)

In [ ]:
# Let's see how much this car actually costs
actual_price = np.expm1(y_test[20])
print('Actual price:    $%.2f' % actual_price)

So our prediction was a bit off — but it's a relatively good prediction given the complexity of the problem. The car costs \$35,000 and we predicted around \$41,500.

So this is how we can first train a full model, and then apply this model to predict the price of a single car.

---


## 2.16 Car Price Prediction Project Summary

This is the last lesson of session two, so let's summarize what we learned. The whole session was one project: predicting the price of a car. Let's walk through it one more time and see how each piece fits together.

### The project

We downloaded a dataset of cars that had prices and different characteristics — things like make, model, engine fuel type, transmission type and others. The target variable we wanted to predict was the price, MSRP (manufacturer's suggested retail price).

### Data preparation and EDA

First we cleaned the dataset and prepared it, so it looks more uniform. The column names had spaces and inconsistent capitalization, and the string values did too, so we made everything lowercase with underscores.

Then we did exploratory data analysis. We identified that the distribution of price has a long tail, and we removed the long tail by applying the **logarithmic transformation**. That's a good idea, because when a distribution has a long tail, machine learning models usually have problems with it — after the transformation it looks like a bell shaped curve.

We also saw that the dataset has missing values, and with missing data we cannot really train a model, so we decided to simply replace them with zeros.

### The validation framework

Then we set up the validation framework: we split the data into training, validation and test parts. This turned out to be one of the most useful things we did — it let us detect problems with the model early.

### Linear regression and the normal equation

Next we looked at how linear regression works. First for a single example, implemented as a simple formula with a for loop. Then we extended it to the vector form using the dot product, and finally to the matrix form — a matrix-vector multiplication.

$$g(X) = Xw \approx y$$

And then we looked at how to actually obtain these weights — how to train the model. We saw that machine learning is not magic: it's just a formula, and this formula is called the **normal equation**:

$$w = (X^T X)^{-1} X^T y$$

We implemented it in NumPy, and with this implementation we trained our first model — the baseline model, which used only the five base numerical features.

The baseline model didn't do really well, as we saw in the graph. But judging from a graph alone is not an objective way to measure the performance of a model. That's why we talked about **RMSE**, the root mean squared error:

$$RMSE = \sqrt{\frac{1}{m} \sum_{i=1}^{m} (g(x_i) - y_i)^2}$$

The `prepare_X` function we wrote gave us the same way of preparing the feature matrix for the training and the validation datasets. That allowed us to experiment a lot faster.

### Feature engineering and categorical variables

After that we did simple **feature engineering** — creating new features from existing ones. We created the `age` feature, and it improved the performance of our model drastically.

Then we looked at how to integrate **categorical variables**. We represented each categorical variable with a bunch of binary columns. This way of encoding categorical variables is called **one-hot encoding**.

After adding all the categorical features we found out that the performance of our model degraded significantly. Luckily we had the validation framework, so we could spot the problem easily. The reason was numerical instability, and as a way to solve it we used **regularization**: we added a small number to the diagonal of the matrix $X^T X$ before inverting it.

**Project summary in one list:**

- EDA — looking at data, finding missing values
- Target variable distribution — long tail, so we applied the **log transformation**
- Validation framework: **train/validation/test split** — it helped us detect problems
- Normal equation — not magic, but math
- Implemented it with NumPy
- **RMSE** to validate our model
- Feature engineering: **age**, categorical features
- **Regularization** to fight numerical instability

### Tuning and the final model

After regularization we tried different values of the regularization parameter to find the best one. We concluded that **0.001** seems like a good choice.

Then we trained our final model: we combined the training and validation datasets into one full train dataset, used our `prepare_X` function again, and trained the model. The prediction we got wasn't far off from the actual price.

### What's next

And in the next section we will talk about **classification**. There, instead of implementing everything ourselves like we did here, we will use a library — scikit-learn. Now that we saw how to implement things ourselves, we are ready to use the library.

---


## 2.17 Explore More

### Questions

- In this project, we included only 5 top features per categorical variable. What happens if we include 10?

> That's not a graded homework, it's just for you if you want to try more things on this project

### Exercise: Try with top 10 features


In [ ]:
# Exercise: Modify the code to use top 10 values per categorical variable
# instead of top 5, and see how RMSE changes.

# Your code here

categories_10 = {}
for c in categorical_variables:
    categories_10[c] = list(df_train[c].value_counts().head(10).index)

def prepare_X_v2(df, cats):
    """Generalized prepare_X that takes a categories dictionary."""
    df = df.copy()
    features = base.copy()

    df['age'] = 2017 - df.year
    features.append('age')

    for v in [2, 3, 4]:
        df['num_doors_%s' % v] = (df.number_of_doors == v).astype('int')
        features.append('num_doors_%s' % v)

    for c, values in cats.items():
        for v in values:
            df['%s_%s' % (c, v)] = (df[c] == v).astype('int')
            features.append('%s_%s' % (c, v))

    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [ ]:
# Compare top-5 vs top-10 features
X_train_10 = prepare_X_v2(df_train, categories_10)
w0_10, w_10 = train_linear_regression_reg(X_train_10, y_train, r=0.001)

X_val_10 = prepare_X_v2(df_val, categories_10)
y_pred_10 = w0_10 + X_val_10.dot(w_10)

print('RMSE with top 5 features:', round(rmse(y_val, w0 + prepare_X(df_val).dot(w)), 4))
print('RMSE with top 10 features:', round(rmse(y_val, y_pred_10), 4))

### Other projects

Here are other datasets you can play with to learn more about the topic:

- [California housing dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) — predict the price of a house
- [Student Performance Data Set](https://archive.ics.uci.edu/ml/datasets/Student+Performance) — predict the performance of students
- [UCI ML Repository](https://archive.ics.uci.edu/ml/datasets.php?task=reg) — contains a lot of other datasets suitable for practicing regression

---

*End of Session 2: Machine Learning for Regression*
